# Phase 2 — FLAN-T5 Translation Training
**Continuous Sign Language Translation** · Phase 2 of 2

Loads the frozen `SemanticEncoder` trained in Phase 1, then fine-tunes `google/flan-t5-small` to map the continuous latent `Z` to English captions.

> ⚠️ **Run Phase 1 first** — this notebook downloads `semantic_encoder.pth` from `mrcyrilgoud/data255`.

**Runtime:** GPU (T4 or better) · **Est. time (100 clips, 2 epochs):** ~4 min


In [1]:
# ── Install dependencies ──────────────────────────────────────────────────────
%pip install -q torch datasets huggingface_hub transformers sentencepiece tqdm numpy


You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cpu


In [3]:
# ── Hugging Face authentication ───────────────────────────────────────────────
# Prefer secrets/env vars. Avoid hardcoding tokens into notebooks.
import os


# Option A (recommended in Colab): Runtime > Manage secrets > add HF_TOKEN
try:
    from google.colab import userdata  # type: ignore
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

# Option B (local/Jupyter): export HF_TOKEN=...
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")

# Fallback: prompt (input hidden)
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)


/Users/mrcyrilgoud/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/mrcyrilgoud/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Model definitions

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SemanticEncoder(nn.Module):
    """1D-CNN: [B, T=60, F=540] → [B, D=512, T'=15]"""
    def __init__(self, input_dim=540, latent_dim=512):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, 256, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(256, 512, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv1d(512, latent_dim, kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm1d(256)
        self.bn2 = nn.BatchNorm1d(512)
        self.bn3 = nn.BatchNorm1d(latent_dim)

    def forward(self, x):          # x: [B, T, F]
        x = x.transpose(1, 2)     # [B, F, T]
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        return x                   # [B, D, T']


class DiffusionDecoder(nn.Module):
    """1D-UNet conditioned on Z; predicts noise."""
    def __init__(self, input_dim=540, latent_dim=512):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(1, 128), nn.GELU(), nn.Linear(128, 128)
        )
        self.upsample_z = nn.ConvTranspose1d(latent_dim, latent_dim, kernel_size=4, stride=4)
        self.net = nn.Sequential(
            nn.Conv1d(input_dim + latent_dim + 128, 512, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(512, 512, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(512, input_dim, kernel_size=3, padding=1),
        )

    def forward(self, x, z, t):
        x = x.transpose(1, 2)                                        # [B, F, T]
        t_emb = self.time_mlp(t.float()).unsqueeze(-1).expand(-1, -1, x.shape[-1])
        z_up  = self.upsample_z(z)
        out   = self.net(torch.cat([x, z_up, t_emb], dim=1))
        return out.transpose(1, 2)                                    # [B, T, F]


class TranslationModel(nn.Module):
    """FLAN-T5-small fed with latent Z [B, 15, 512] as encoder embeddings."""
    def __init__(self):
        super().__init__()
        from transformers import T5ForConditionalGeneration
        self.t5 = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

    def forward(self, z, labels=None):
        return self.t5(inputs_embeds=z, labels=labels) if labels is not None                else self.t5.generate(inputs_embeds=z)


## Dataset — streaming with on-the-fly feature engineering

In [5]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import IterableDataset

# 15 face landmarks capturing Non-Manual Markers (eyebrows, eye corners, nose, mouth)
FACE_LANDMARK_IDXS = [70, 105, 336, 300, 33, 133, 362, 263, 4, 61, 291, 13, 14, 17, 0]
_POSE_SLICE  = slice(0,   33)
_FACE_SLICE  = slice(33,  501)
_LHAND_SLICE = slice(501, 522)
_RHAND_SLICE = slice(522, 543)
T_WINDOW, T_STRIDE = 60, 30


def engineer_features(raw: np.ndarray):
    """raw: [T, 543, 3]  →  torch.Tensor [T, 540]"""
    T = raw.shape[0]
    if T < 2:
        return None
    pose  = raw[:, _POSE_SLICE,  :]   # [T, 33, 3]
    face  = raw[:, _FACE_SLICE,  :]   # [T, 468, 3]
    lhand = raw[:, _LHAND_SLICE, :]   # [T, 21, 3]
    rhand = raw[:, _RHAND_SLICE, :]   # [T, 21, 3]

    centre = pose.mean(axis=1, keepdims=True)  # body-centre per frame
    pose, face, lhand, rhand = (a - centre for a in (pose, face, lhand, rhand))

    kpts  = np.concatenate([pose, face[:, FACE_LANDMARK_IDXS, :], lhand, rhand], axis=1)  # [T,90,3]
    delta = np.zeros_like(kpts)
    delta[1:] = kpts[1:] - kpts[:-1]
    features = np.concatenate([kpts.reshape(T, -1), delta.reshape(T, -1)], axis=1)        # [T,540]
    return torch.from_numpy(features.astype(np.float32))


def sliding_windows(feat, window=T_WINDOW, stride=T_STRIDE):
    T = feat.shape[0]
    if T < window:
        yield F.pad(feat, (0, 0, 0, window - T))
    else:
        start = 0
        while start + window <= T:
            yield feat[start:start + window]
            start += stride


class RealSignLanguageDataset(IterableDataset):
    """Streams from bdanko/how2sign-landmarks-front-raw-parquet on Hugging Face."""
    def __init__(self, split="train", max_samples=None,
                 repo_id="bdanko/how2sign-landmarks-front-raw-parquet"):
        self.split, self.max_samples, self.repo_id = split, max_samples, repo_id

    def __iter__(self):
        from datasets import load_dataset
        ds = load_dataset(self.repo_id, split=self.split, streaming=True)
        count = 0
        for sample in ds:
            if self.max_samples and count >= self.max_samples:
                break
            raw = np.frombuffer(sample["features"], dtype=np.float32).reshape(sample["shape"])
            feat = engineer_features(raw)
            if feat is None:
                continue
            for chunk in sliding_windows(feat):
                yield chunk, sample.get("sentence", "")
            count += 1


## ⚙️ Configuration


In [6]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
ENCODER_REPO = "mrcyrilgoud/data255"
TRANS_REPO   = "mrcyrilgoud/data255"
SPLIT        = "train"
MAX_SAMPLES  = 100     # set to None for full dataset
BATCH_SIZE   = 8
LR           = 5e-5
EPOCHS       = 2
CKPT_DIR     = "/tmp/phase2_ckpt"


## Load & freeze the Semantic Encoder

In [7]:
import torch
from huggingface_hub import hf_hub_download

encoder = SemanticEncoder().to(device)
weights = hf_hub_download(repo_id=ENCODER_REPO, filename="semantic_encoder.pth",
                          token=HF_TOKEN)
encoder.load_state_dict(torch.load(weights, map_location=device))
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

print("Semantic Encoder loaded and frozen ✓")


Semantic Encoder loaded and frozen ✓


## Training loop

In [8]:
import os, torch, torch.optim as optim
from transformers import T5Tokenizer
from tqdm.auto import tqdm
from huggingface_hub import HfApi, create_repo, upload_folder

os.makedirs(CKPT_DIR, exist_ok=True)

translation_model = TranslationModel().to(device)
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
optimizer = optim.AdamW(translation_model.parameters(), lr=LR)

dataset = RealSignLanguageDataset(split=SPLIT, max_samples=MAX_SAMPLES)

for epoch in range(EPOCHS):
    total_loss, count = 0.0, 0
    batch_z, batch_labels = [], []

    for features, sentence in dataset:
        with torch.no_grad():
            z = encoder(features.unsqueeze(0).to(device))   # [1, D, T']
        batch_z.append(z.squeeze(0).transpose(0, 1))        # [T', D]
        batch_labels.append(sentence)

        if len(batch_z) < BATCH_SIZE:
            continue

        z_tensor = torch.stack(batch_z).to(device)          # [B, T', D]
        labels   = tokenizer(
            batch_labels, return_tensors="pt",
            padding=True, truncation=True, max_length=128
        ).input_ids.to(device)

        outputs = translation_model(z_tensor, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item(); count += 1
        batch_z, batch_labels = [], []

    avg = total_loss / max(count, 1)
    print(f"Epoch {epoch+1}/{EPOCHS}  avg_loss={avg:.4f}")

    # Save & upload after every epoch
    torch.save(translation_model.state_dict(), f"{CKPT_DIR}/translation_model.pth")
    create_repo(TRANS_REPO, token=HF_TOKEN, exist_ok=True)
    upload_folder(
        folder_path=CKPT_DIR,
        repo_id=TRANS_REPO,
        token=HF_TOKEN,
        commit_message=f"Phase 2 epoch {epoch+1}/{EPOCHS}",
    )
    print(f"  → checkpoint uploaded ✓")

print("Training complete.")


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Repo card metadata block was not found. Setting CardData to empty.


Epoch 1/2  avg_loss=8.4085


Processing Files (1 / 1): 100%|██████████|  308MB /  308MB, 2.24MB/s  
New Data Upload: 100%|██████████|  308MB /  308MB, 2.24MB/s  


  → checkpoint uploaded ✓


Repo card metadata block was not found. Setting CardData to empty.


Epoch 2/2  avg_loss=4.5985


Processing Files (1 / 1): 100%|██████████|  308MB /  308MB, 4.01MB/s  
New Data Upload: 100%|██████████|  308MB /  308MB, 4.01MB/s  


  → checkpoint uploaded ✓
Training complete.
